<a href="https://colab.research.google.com/github/shivaprajapati34390-netizen/ML-project/blob/main/Document_Analysis_using_LLMs_with_Python.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

step 1 extract step from data


In [2]:
!pip install pdfplumber
import pdfplumber

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.1/68.1 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 80.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 88.1 MB/s eta 0:00:00


In [3]:
pdf_path=pdfplumber.open("/content/sample_data/google_terms_of_service_en_in.pdf")

In [4]:
output_text_file= "extracted_text.text"

In [5]:
with pdf_path as pdf:
  extracted_text=""
  for page in pdf.pages:
    extracted_text +=page.extract_text()

In [6]:
with open(output_text_file,"w")as text_file:
    text_file.write(extracted_text)

In [7]:

print(f"text extracted and saved to {output_text_file}")

text extracted and saved to extracted_text.text


step 2 Preview the extracted text




In [8]:
# reading the pdf content
with open(output_text_file, "r") as file:
    document_text = file.read()

In [9]:
# Preview the document content

In [10]:
print(document_text[:500])

GOOGLE TERMS OF SERVICE
Effective May 22, 2024 | Archived versions
What’s covered in these terms
We know it’s tempting to skip these Terms of
Service, but it’s important to establish what you
can expect from us as you use Google services,
and what we expect from you.
These Terms of Service re ect the way Google’s business works, the laws that apply to
our company, and certain things we’ve always believed to be true. As a result, these Terms
of Service help de ne Google’s relationship with you as


step 3 summarize the document

In [11]:
!pip install transformers
from transformers import pipeline

In [13]:
from transformers import pipeline

text_generator = pipeline("text-generation", model='gpt2')

#  summarize the document text (using text-generation for abstractive summarization)
# Note: 'extractive_summary' variable is not defined, using 'document_text' as input for now.
# Please ensure 'extractive_summary' is defined if you intend to use it.
abstractive_summary = text_generator(document_text[:1000], max_length=150, num_return_sequences=1, do_sample=False)
print("Abstractive Summary:", abstractive_summary[0]['generated_text'])

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'num_return_sequences', 'max_length', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=150) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Abstractive Summary: GOOGLE TERMS OF SERVICE
Effective May 22, 2024 | Archived versions
What’s covered in these terms
We know it’s tempting to skip these Terms of
Service, but it’s important to establish what you
can expect from us as you use Google services,
and what we expect from you.
These Terms of Service re ect the way Google’s business works, the laws that apply to
our company, and certain things we’ve always believed to be true. As a result, these Terms
of Service help de ne Google’s relationship with you as you interact with our services. For
example, these terms include the following topic headings:
What you can expect from us, which describes how we provide and develop our
services
What we expect from you, which establishes certain rules for using our services
Content in Google services, which describes the intellectual property rights to the
content you  nd in our services — whether that content belongs to you, Google, or
others
In case of problems or disagreements, which d

step 4 split the documnet into pages and sentences

In [16]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab') # Download the missing resource
from nltk.tokenize import sent_tokenize

# split text into sentences

sentences = sent_tokenize(document_text)

#  combine sentences into passages
passages=[]
current_passage=""
max_passage_length=400

for sentence in sentences:
  if len(current_passage.split())+len(sentence.split()) <=max_passage_length:
    current_passage+=" "+ sentence
  else:
    passages.append(current_passage.strip())
    current_passage = sentence

if current_passage:
  passages.append(current_passage.strip())

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


step 5 Genarate question from the passage using LLM

In [24]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Load tokenizer and model directly for question generation
tokenizer = AutoTokenizer.from_pretrained("valhalla/t5-base-e2e-qg")
qg_model= AutoModelForSeq2SeqLM.from_pretrained("valhalla/t5-base-e2e-qg")

# function to generate questions
def generate_question_from_passage(passage, min_questions=3):
  input_text=f"generate questions: {passage}"
  inputs = tokenizer(input_text, return_tensors="pt", max_length=512, truncation=True)
  outputs = qg_model.generate(inputs["input_ids"], max_length=64, num_beams=4, early_stopping=True)
  generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
  questions = [q.strip() for q in generated_text.split('<sep>') if q.strip()]

  # If fewer than min_questions, try to regenerate from smaller parts (simplified for this fix)
  if len(questions) < min_questions:
    # For simplicity, we'll just return what we have. More complex regeneration logic can be added.
    pass

  return questions[:min_questions] # return only up to min_questions


Loading weights:   0%|          | 0/260 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [28]:
# generate questions from passages
for idx, passage in enumerate(passages):
    questions = generate_question_from_passage(passage)
    print(f"Passage {idx+1}:\n{passage}\n")
    print("Generated Questions:")
    for q in questions:
        print(f"- {q}")
    print(f"\n{'-'*50}\n")

Passage 1:
GOOGLE TERMS OF SERVICE
Effective May 22, 2024 | Archived versions
What’s covered in these terms
We know it’s tempting to skip these Terms of
Service, but it’s important to establish what you
can expect from us as you use Google services,
and what we expect from you. These Terms of Service re ect the way Google’s business works, the laws that apply to
our company, and certain things we’ve always believed to be true. As a result, these Terms
of Service help de ne Google’s relationship with you as you interact with our services. For
example, these terms include the following topic headings:
What you can expect from us, which describes how we provide and develop our
services
What we expect from you, which establishes certain rules for using our services
Content in Google services, which describes the intellectual property rights to the
content you  nd in our services — whether that content belongs to you, Google, or
others
In case of problems or disagreements, which describes o


step 5 answer the genarated question  using  QA model

In [29]:
#  load the qa_pipeline
qa_pipeline=pipeline("question-answering",model="deepset/roberta-base-squad2")


config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/496M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RobertaForQuestionAnswering LOAD REPORT from: deepset/roberta-base-squad2
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/79.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

In [32]:
#  function to track answer only unique question
def answer_unique_question(passages,qa_pipeline):
  answerd_question=set() # to store unique question

  for idx,passage in enumerate(passages):
    questions=generate_question_from_passage(passage)

    for question in questions:
      if question not in answerd_question: # check if the question is already answerd
        answer=qa_pipeline(question=question,context=passage)
        print(f"Q{question}")
        print(f"A{answer['answer']}\n")
        answerd_question.add(question) # add the question to the set to avoid repetition

    print(f"{'='*50}\n")

answer_unique_question(passages,qa_pipeline)

QWhat do these Terms of Service describe?
Athe way Google’s business works

QWhat do these Terms of Service help dene the relationship between you and Google?
AYour relationship with GoogleThese

QWho is responsible for your child's activity on the services?
AIf you’re a parent or legal guardian


QWhat are two examples of services that are subject to these terms?
ASearch and Maps

QWhat are two examples of devices that are subject to these terms?
AGoogle Nest and Pixel

QWhat do Google services work together to make it easier for you to move from one activity to the next?
Adevices


QWhat are some of the resources available from our policies site?
APrivacy Policy, Copyright Help Center, Safety Center, Transparency
Center

QWhat are some of the warnings that we may provide within our services?
Adialog boxes that alert you to
important information

QWhat are some of the basic rules of conduct?
Acomply with applicable laws


QWhat are some ways that you must not abuse, harm, interfere wi